
# IFS: 5 konkret empfohlene Visualisierungen für einen Management-Bericht

Dieses Notebook zeigt ein **kompaktes Set aus 5 empfehlenswerten Visualisierungen** für einen Bericht über das IFS (IBM i / AS400) auf Basis von Dateimetadaten.

## Zielgruppe
Das Konzept ist auf eine **management- bzw. vorstandstaugliche Darstellung** ausgelegt:
- nicht die technische Datei-Einzelansicht,
- sondern eine **Verdichtung auf Umfang, Struktur, Speicherverbrauch und Altbestände**.

## Enthaltene Visualisierungen
1. **Speicherverbrauch nach Hauptverzeichnis**  
2. **Pareto-Diagramm des Speicherverbrauchs**  
3. **Dateianzahl nach Hauptverzeichnis**  
4. **Altersstruktur nach Hauptverzeichnis**  
5. **Heatmap: Verzeichnis × Altersklasse**

## Datengrundlage
Das Notebook arbeitet mit **Dummy-Daten**, die an folgende Spalten angelehnt sind:

- `Name`
- `Path`
- `Type`
- `INode`
- `ID_Nbr`
- `nbrLinks`
- `UID`
- `GrID`
- `Size`
- `MRAcc_Tme`
- `MRMod_Tme`
- `MRCr_Tme`

## Späterer Praxiseinsatz
Sobald echte CSV-Daten aus Deinem IFS-Scan vorliegen, kannst Du den Abschnitt **"1) Dummy-Daten erzeugen"** durch das Einlesen Deiner Datei ersetzen.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import PurePosixPath
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

rng = np.random.default_rng(42)


## 1) Dummy-Daten erzeugen

In [ ]:

root_dirs = {
    "/ifs/finance": {
        "subdirs": ["reports", "exports", "archive", "monthly"],
        "weight": 0.20,
        "exts": [".csv", ".xlsx", ".pdf", ".txt"]
    },
    "/ifs/sales": {
        "subdirs": ["daily", "weekly", "archive", "broker"],
        "weight": 0.18,
        "exts": [".csv", ".xlsx", ".pdf", ".log"]
    },
    "/ifs/hr": {
        "subdirs": ["payroll", "archive", "contracts", "exports"],
        "weight": 0.10,
        "exts": [".pdf", ".csv", ".txt", ""]
    },
    "/ifs/logs": {
        "subdirs": ["batch", "jobs", "archive", "system"],
        "weight": 0.16,
        "exts": [".log", ".txt", ".csv", ""]
    },
    "/ifs/interfaces": {
        "subdirs": ["inbound", "outbound", "archive", "temp"],
        "weight": 0.14,
        "exts": [".csv", ".xml", ".json", ".txt"]
    },
    "/ifs/documents": {
        "subdirs": ["board", "policies", "archive", "manuals"],
        "weight": 0.12,
        "exts": [".pdf", ".docx", ".xlsx", ".pptx"]
    },
    "/ifs/projects": {
        "subdirs": ["legacy", "analysis", "archive", "deliverables"],
        "weight": 0.10,
        "exts": [".ipynb", ".py", ".csv", ".pdf"]
    },
}

n_files = 2500
root_choices = list(root_dirs.keys())
root_weights = np.array([root_dirs[k]["weight"] for k in root_choices], dtype=float)
root_weights = root_weights / root_weights.sum()

uids = [1001, 1002, 1003, 1100, 1200]
gids = [200, 210, 220, 230]
devices = [1, 2, 3]

def random_timestamp(days_back_low=0, days_back_high=3650):
    days_back = int(rng.integers(days_back_low, days_back_high))
    secs = int(rng.integers(0, 24 * 3600))
    return pd.Timestamp.now().floor("s") - pd.Timedelta(days=days_back, seconds=secs)

rows = []
for i in range(n_files):
    root = rng.choice(root_choices, p=root_weights)
    subdir = rng.choice(root_dirs[root]["subdirs"])
    ext = rng.choice(root_dirs[root]["exts"])

    if root == "/ifs/logs":
        size_bytes = int(np.exp(rng.normal(11.0, 1.1)))
    elif root in ["/ifs/finance", "/ifs/sales"]:
        size_bytes = int(np.exp(rng.normal(13.5, 1.2)))
    elif root == "/ifs/documents":
        size_bytes = int(np.exp(rng.normal(14.0, 1.0)))
    else:
        size_bytes = int(np.exp(rng.normal(12.5, 1.3)))

    size_bytes = max(size_bytes, 512)

    if rng.random() < 0.01:
        size_bytes *= int(rng.integers(100, 500))

    mtime = random_timestamp(0, 3650)
    atime = mtime + pd.Timedelta(days=int(rng.integers(0, 120)))
    ctime = mtime - pd.Timedelta(days=int(rng.integers(0, 60)))

    name = f"file_{i:05d}{ext}"
    path = f"{root}/{subdir}/{name}"

    rows.append({
        "Name": name,
        "Path": path,
        "Type": int(rng.choice([33188, 33261, 33204])),
        "INode": int(rng.integers(10_000, 9_999_999)),
        "ID_Nbr": int(rng.choice(devices)),
        "nbrLinks": int(rng.choice([1, 1, 1, 2, 3])),
        "UID": int(rng.choice(uids)),
        "GrID": int(rng.choice(gids)),
        "Size": int(size_bytes),
        "MRAcc_Tme": atime,
        "MRMod_Tme": mtime,
        "MRCr_Tme": ctime,
    })

df = pd.DataFrame(rows)
df.head()


In [ ]:

# Optional: Dummy-Daten zusätzlich als CSV speichern
df.to_csv(r"/mnt/data/ifs_dummy_scan_fuer_5_visualisierungen.csv", sep=";", encoding="utf-8", index=False)
print("CSV geschrieben nach:", r"/mnt/data/ifs_dummy_scan_fuer_5_visualisierungen.csv")


## 2) Datenaufbereitung

In [ ]:

for col in ["MRAcc_Tme", "MRMod_Tme", "MRCr_Tme"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

def get_root_dir(path: str) -> str:
    p = PurePosixPath(path)
    return "/" + "/".join(p.parts[1:3]) if len(p.parts) >= 3 else str(p)

def get_subdir(path: str) -> str:
    p = PurePosixPath(path)
    return "/" + "/".join(p.parts[1:4]) if len(p.parts) >= 4 else str(p)

def get_ext(name: str) -> str:
    suffix = PurePosixPath(name).suffix.lower()
    return suffix if suffix else "[ohne Endung]"

df["root_dir"] = df["Path"].map(get_root_dir)
df["sub_dir"] = df["Path"].map(get_subdir)
df["ext"] = df["Name"].map(get_ext)
df["size_mb"] = df["Size"] / (1024**2)
df["size_gb"] = df["Size"] / (1024**3)

today = pd.Timestamp.now().floor("D")
df["age_days"] = (today - df["MRMod_Tme"].dt.floor("D")).dt.days

age_bins = [-1, 30, 180, 365, 3*365, 10_000]
age_labels = ["<= 30 Tage", "31-180 Tage", "181-365 Tage", "1-3 Jahre", "> 3 Jahre"]
df["age_class"] = pd.cut(df["age_days"], bins=age_bins, labels=age_labels)

display(df.sample(5, random_state=1))



## 3) Kurzüberblick / Executive Summary

Die folgenden Kennzahlen helfen, den Umfang des Scans kurz einzuordnen.  
Sie sind **nicht** Teil der fünf Kernvisualisierungen, aber eine sinnvolle Einstiegssicht.


In [ ]:

kpis = {
    "Gesamtdateien": len(df),
    "Gesamtvolumen (GB)": round(df["size_gb"].sum(), 2),
    "Anzahl Hauptverzeichnisse": df["root_dir"].nunique(),
    "Größte Einzeldatei (MB)": round(df["size_mb"].max(), 2),
    "Anteil > 3 Jahre alt (%)": round((df["age_class"] == "> 3 Jahre").mean() * 100, 1),
}

display(pd.DataFrame({"Kennzahl": list(kpis.keys()), "Wert": list(kpis.values())}))



---

# Die 5 konkret empfohlenen Visualisierungen

Jede Visualisierung enthält:
- eine **Management-Frage**
- eine kurze **Interpretation**
- den zugehörigen Python-Code



## Visualisierung 1: Speicherverbrauch nach Hauptverzeichnis

**Management-Frage:**  
Wo liegt der Speicherverbrauch im IFS?

**Warum diese Grafik wichtig ist:**  
Sie ist meist die **einfachste und stärkste Einstiegsgrafik**.  
Man erkennt sofort, welche Bereiche das Gesamtvolumen dominieren.

**Typische Botschaft:**  
„Wenige Hauptverzeichnisse tragen den Hauptanteil des Speicherverbrauchs.“


In [ ]:

size_by_root = (
    df.groupby("root_dir", as_index=False)["size_gb"]
      .sum()
      .sort_values("size_gb", ascending=True)
)

plt.figure(figsize=(10, 5))
plt.barh(size_by_root["root_dir"], size_by_root["size_gb"])
plt.xlabel("Gesamtgröße in GB")
plt.ylabel("Hauptverzeichnis")
plt.title("Speicherverbrauch nach Hauptverzeichnis")
plt.tight_layout()
plt.show()

display(size_by_root.sort_values("size_gb", ascending=False))



## Visualisierung 2: Pareto-Diagramm des Speicherverbrauchs

**Management-Frage:**  
Wie stark konzentriert sich der Speicherverbrauch auf wenige Verzeichnisse?

**Warum diese Grafik wichtig ist:**  
Der Pareto-Ansatz macht sehr sichtbar, ob das bekannte **80/20-Prinzip** greift:
ein kleiner Teil der Verzeichnisse erklärt den Großteil des Volumens.

**Typische Botschaft:**  
„Bereits die größten X Verzeichnisse erklären Y % des gesamten Volumens.“


In [ ]:

pareto_df = (
    df.groupby("root_dir", as_index=False)["size_gb"]
      .sum()
      .sort_values("size_gb", ascending=False)
      .reset_index(drop=True)
)

pareto_df["cum_pct"] = pareto_df["size_gb"].cumsum() / pareto_df["size_gb"].sum() * 100

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(pareto_df["root_dir"], pareto_df["size_gb"])
ax1.set_xlabel("Hauptverzeichnis")
ax1.set_ylabel("Größe in GB")
ax1.set_title("Pareto-Diagramm des Speicherverbrauchs")
ax1.tick_params(axis="x", rotation=30)

ax2 = ax1.twinx()
ax2.plot(pareto_df["root_dir"], pareto_df["cum_pct"], marker="o")
ax2.set_ylabel("Kumulierter Anteil in %")
ax2.set_ylim(0, 110)

plt.tight_layout()
plt.show()

display(pareto_df)



## Visualisierung 3: Dateianzahl nach Hauptverzeichnis

**Management-Frage:**  
Wo liegen besonders viele Dateien?

**Warum diese Grafik wichtig ist:**  
Ein Bereich kann entweder
- **viele kleine Dateien** oder
- **wenige große Dateien**

enthalten.  
Die Dateianzahl ergänzt deshalb die reine Volumensicht sehr sinnvoll.

**Typische Botschaft:**  
„Die mengenstarken Verzeichnisse sind nicht zwingend dieselben wie die volumenstarken Verzeichnisse.“


In [ ]:

count_by_root = (
    df.groupby("root_dir", as_index=False)["Name"]
      .count()
      .rename(columns={"Name": "file_count"})
      .sort_values("file_count", ascending=True)
)

plt.figure(figsize=(10, 5))
plt.barh(count_by_root["root_dir"], count_by_root["file_count"])
plt.xlabel("Anzahl Dateien")
plt.ylabel("Hauptverzeichnis")
plt.title("Dateianzahl nach Hauptverzeichnis")
plt.tight_layout()
plt.show()

display(count_by_root.sort_values("file_count", ascending=False))



## Visualisierung 4: Altersstruktur nach Hauptverzeichnis

**Management-Frage:**  
Welche Bereiche sind aktuell aktiv, und wo liegen Altbestände?

**Warum diese Grafik wichtig ist:**  
Mit dieser Sicht kann man Altbestände von operativen Bereichen trennen.
Das ist besonders hilfreich für Diskussionen zu:
- Archivierung
- Bereinigung
- Verantwortlichkeiten
- Daten-Governance

**Typische Botschaft:**  
„Bestimmte Verzeichnisse weisen einen hohen Anteil älterer, lange nicht geänderter Dateien auf.“


In [ ]:

age_pivot = (
    df.pivot_table(index="root_dir", columns="age_class", values="Name", aggfunc="count", fill_value=0)
      .reindex(columns=age_labels)
)

age_pivot_pct = age_pivot.div(age_pivot.sum(axis=1), axis=0) * 100

age_pivot_pct.plot(kind="bar", stacked=True, figsize=(11, 6))
plt.xlabel("Hauptverzeichnis")
plt.ylabel("Anteil in %")
plt.title("Altersstruktur nach Hauptverzeichnis")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

display(age_pivot)



## Visualisierung 5: Heatmap Verzeichnis × Altersklasse

**Management-Frage:**  
In welchen Verzeichnissen konzentrieren sich alte bzw. junge Dateien besonders stark?

**Warum diese Grafik wichtig ist:**  
Die Heatmap verdichtet sehr viele Informationen in einer einzigen Übersicht.
Sie eignet sich gut, um **Muster, Konzentrationen und Ausreißer** schnell zu erkennen.

**Typische Botschaft:**  
„Altbestände konzentrieren sich auf bestimmte Bereiche und sind nicht gleichmäßig über das IFS verteilt.“


In [ ]:

heatmap_df = (
    df.pivot_table(index="root_dir", columns="age_class", values="Size", aggfunc="sum", fill_value=0)
      .reindex(columns=age_labels)
)

heatmap_gb = heatmap_df / (1024**3)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(heatmap_gb.values, aspect="auto")

ax.set_xticks(range(len(heatmap_gb.columns)))
ax.set_xticklabels(heatmap_gb.columns, rotation=30, ha="right")
ax.set_yticks(range(len(heatmap_gb.index)))
ax.set_yticklabels(heatmap_gb.index)

ax.set_title("Heatmap: Verzeichnis × Altersklasse (GB)")
ax.set_xlabel("Altersklasse")
ax.set_ylabel("Hauptverzeichnis")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Größe in GB")

plt.tight_layout()
plt.show()

display(heatmap_gb.round(2))



## 4) Zusammenfassung für den Bericht

Wenn Du nur **eine kleine, überzeugende Berichtsstrecke** bauen willst, ist dieses 5er-Set sehr gut geeignet:

1. **Speicherverbrauch nach Hauptverzeichnis**  
   → zeigt das Grundbild

2. **Pareto-Diagramm**  
   → zeigt die Konzentration des Problems

3. **Dateianzahl nach Hauptverzeichnis**  
   → trennt Mengen- von Volumenproblemen

4. **Altersstruktur**  
   → zeigt operative Bereiche vs. Altbestände

5. **Heatmap**  
   → verdichtet Muster und Auffälligkeiten

### Praktische Management-Botschaften
- Der Speicherverbrauch konzentriert sich auf wenige Hauptverzeichnisse.
- Mengen- und Volumenschwerpunkte fallen nicht zwingend zusammen.
- Altbestände sind erkennbar und lassen sich fachlich priorisieren.
- Der Pareto-Effekt macht Prioritäten für weitere Prüfungen sichtbar.



## 5) Umstellung auf echte IFS-Daten

Sobald Deine echten Daten vorliegen, kannst Du hier statt der Dummy-Daten Deine CSV laden, zum Beispiel:

```python
df = pd.read_csv("ifs_scan.csv", sep=";", encoding="utf-8")
```

Danach führst Du nur noch die Zellen ab **"2) Datenaufbereitung"** aus.
